# Elliptic++ — Initial Experiments

Classify Bitcoin **transactions** and **wallets** as illicit / licit. Dataset: [git-disl/EllipticPlusPlus](https://github.com/git-disl/EllipticPlusPlus), which extends the original Elliptic dataset (Weber et al. 2019) with a wallet-level ("actors") graph.

## Data structure

### Transactions sub-dataset (`transactions_data/`)
Single-graph view: each node is a Bitcoin transaction.

| File | Role | Shape |
|---|---|---|
| `txs_features.csv` | node features | 203,769 × 184 — `txId`, `Time step`, 93 *Local* features (per-tx), 72 *Aggregate* features (1-hop neighbour summaries), 17 metadata fields (BTC totals, fees, in/out degree, …) |
| `txs_classes.csv` | node labels | 203,769 × 2 — `class` 1 = illicit (4,545), 2 = licit (42,019), 3 = unknown (157,205) |
| `txs_edgelist.csv` | edges | 234,355 directed `txId1 → txId2`: an output of `txId1` is consumed as an input of `txId2` (BTC flow between transactions) |

49 time steps (~biweekly snapshots). Each tx belongs to exactly one time step.

### Wallets / actors sub-dataset (`actors_data/`)
Heterogeneous graph with two node types: wallet addresses and transactions.

| File | Role | Shape |
|---|---|---|
| `wallets_features.txt` | wallet features per time step | 1,268,260 × 57 — `address`, `Time step`, 55 numeric features. Same address has one row per time step in which it is active |
| `wallets_classes.txt` | per-address label | 822,942 × 2 — `class` 1 = illicit (14,266), 2 = licit (251,088), 3 = unknown (557,588) |
| `AddrAddr_edgelist.txt` | wallet → wallet | ~2.87 M directed edges from input address to output address (co-spending / direct interaction) |
| `AddrTx_edgelist.txt` | wallet → tx | ~1.31 M edges: that address supplied an input to that tx |
| `TxAddr_edgelist.txt` | tx → wallet | ~1.27 M edges: that tx paid an output to that address |

Composing `AddrTx ∘ TxAddr` recovers wallet → wallet money-flow paths through explicit tx nodes — strictly more expressive than `AddrAddr` alone.

## Train / val / test split (strict chronological)

- **Train**: time steps 1–29
- **Val**: time steps 30–34
- **Test**: time steps 35–49 ("the cliff")

The post-34 window contains a documented distribution shift (a major dark-market shutdown around step 43) that has been shown to collapse illicit-class F1 of models trained on the pre-34 window. Carving val out of the pre-cliff range lets us tune hyperparameters without peeking at the post-cliff data, and makes the in-distribution vs. cliff gap visible.

Labels are mapped to `label ∈ {1: illicit, 0: licit, -1: unknown}`. Rows with `label = -1` are excluded from supervised loss and from metrics, but kept in the graph so they can still participate in message passing as unlabelled nodes.

## Models
1. **Random Forest** baselines on tx and wallet tabular features.
2. **GCN, GraphSAGE, GAT** on the tx graph (transductive node classification).
3. **Temporal Transformer** with self-attention over each wallet's per-time-step feature trajectory.

In [22]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
torch.manual_seed(42)
np.random.seed(42)

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "transactions_data").exists():
    ROOT = ROOT.parent
TRANSACTIONS_DIR = ROOT / "transactions_data"
WALLETS_DIR = ROOT / "actors_data"

TRAIN_END = 29
VAL_END = 34
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"root={ROOT}  device={DEVICE}")

root=/Users/jarayliu/Documents/GitHub/stat175-final  device=cpu


## 1. Load data (single cell)

Add `label ∈ {1 illicit, 0 licit, -1 unknown}` alongside the original `class` column. We do **not** drop unknown rows — they remain available as unlabelled nodes for graph models. Filtering with `label != -1` happens later at the modelling step.

In [23]:
transactions_features = pd.read_csv(TRANSACTIONS_DIR / "txs_features.csv")
transactions_classes = pd.read_csv(TRANSACTIONS_DIR / "txs_classes.csv")
transactions_edges = pd.read_csv(TRANSACTIONS_DIR / "txs_edgelist.csv")

transactions_classes["label"] = transactions_classes["class"].map({1: 1, 2: 0, 3: -1}).astype(np.int8)
transactions_df = transactions_features.merge(
    transactions_classes[["txId", "class", "label"]], on="txId", how="left"
)

transactions_df["label"] = transactions_df["label"].fillna(-1).astype(np.int8)
transactions_feature_cols = [c for c in transactions_df.columns if c not in ("txId", "Time step", "class", "label")]
# 17 metadata cols (in/out BTC, fees, size, ...) have 965 missing rows — coinbase / special txs.
# Without imputation, StandardScaler propagates NaN through every node and the GNN loss is NaN.
transactions_df[transactions_feature_cols] = (
    transactions_df[transactions_feature_cols].fillna(0).astype(np.float32)
)

with open(WALLETS_DIR / "wallets_features.txt") as f:
    wallet_columns = f.readline().rstrip("\n").split(",")

wallet_dtype = {c: np.float32 for c in wallet_columns if c not in ("address", "Time step")}
wallet_dtype["address"] = "string"
wallet_dtype["Time step"] = np.int16
wallets_features = pd.read_csv(WALLETS_DIR / "wallets_features.txt", dtype=wallet_dtype)
wallets_classes = pd.read_csv(WALLETS_DIR / "wallets_classes.txt")
wallets_classes["label"] = wallets_classes["class"].map({1: 1, 2: 0, 3: -1}).astype(np.int8)
wallets_df = wallets_features.merge(
    wallets_classes[["address", "class", "label"]], on="address", how="left"
)

wallets_df["label"] = wallets_df["label"].fillna(-1).astype(np.int8)
wallets_feature_cols = [c for c in wallets_df.columns if c not in ("address", "Time step", "class", "label")]

addr_addr_edges = pd.read_csv(WALLETS_DIR / "AddrAddr_edgelist.txt")
addr_tx_edges = pd.read_csv(WALLETS_DIR / "AddrTx_edgelist.txt")
tx_addr_edges = pd.read_csv(WALLETS_DIR / "TxAddr_edgelist.txt")

print(f"transactions rows={len(transactions_df):>9,} features={len(transactions_feature_cols):>3}  edges={len(transactions_edges):>9,}")
print(f"wallets rows={len(wallets_df):>9,} features={len(wallets_feature_cols):>3}  unique addrs={wallets_df['address'].nunique():>9,}")
print(f"wallet edges  AddrAddr={len(addr_addr_edges):>9,} AddrTx={len(addr_tx_edges):>9,}  TxAddr={len(tx_addr_edges):>9,}")
print("\ntransactions label counts:", transactions_df["label"].value_counts().to_dict())
print("wallets label counts (per row):", wallets_df["label"].value_counts().to_dict())
print(f"\nNaN audit  transactions features={int(transactions_df[transactions_feature_cols].isna().sum().sum())} wallet features={int(wallets_df[wallets_feature_cols].isna().sum().sum())}")

transactions rows=  203,769 features=182  edges=  234,355
wallets rows=1,268,260 features= 55  unique addrs=  822,942
wallet edges  AddrAddr=2,868,964 AddrTx=  477,117  TxAddr=  837,124

transactions label counts: {-1: 157205, 0: 42019, 1: 4545}
wallets label counts (per row): {-1: 900788, 0: 338871, 1: 28601}

NaN audit  transactions features=0 wallet features=0


## 2. Random Forest baselines

Tabular features only — no graph signal. Strict chronological 3-way split, class-balanced trees, illicit-class metrics on val and test separately.

In [24]:
def random_forest_temporal(df, feature_cols, time_step_col="Time step", n_estimators=100, sample=None):
    labelled = df[df["label"] != -1]

    if sample is not None and len(labelled) > sample:
        labelled = labelled.sample(n=sample, random_state=42)
        
    train_df = labelled[labelled[time_step_col] <= TRAIN_END]
    val_df = labelled[(labelled[time_step_col] > TRAIN_END) & (labelled[time_step_col] <= VAL_END)]
    test_df = labelled[labelled[time_step_col] > VAL_END]

    X_train, y_train = train_df[feature_cols].values, train_df["label"].values
    X_val, y_val = val_df[feature_cols].values, val_df["label"].values
    X_test, y_test = test_df[feature_cols].values, test_df["label"].values

    print(f"TRAIN (1-{TRAIN_END}): {len(y_train):>9,} rows | illicit={int(y_train.sum()):>6,} ({y_train.mean()*100:5.2f}%)")
    print(f"VAL ({TRAIN_END+1}-{VAL_END}): {len(y_val):>9,} rows | illicit={int(y_val.sum()):>6,} ({y_val.mean()*100:5.2f}%)")
    print(f"TEST ({VAL_END+1}-49): {len(y_test):>9,} rows | illicit={int(y_test.sum()):>6,} ({y_test.mean()*100:5.2f}%)")

    classifier = RandomForestClassifier(
        n_estimators=n_estimators, class_weight="balanced",
        n_jobs=-1, random_state=42,
    )
    classifier.fit(X_train, y_train)

    print("\nVAL performance:")
    print(classification_report(y_val, classifier.predict(X_val), target_names=["licit", "illicit"], digits=4, zero_division=0))

    print("TEST performance:")
    print(classification_report(y_test, classifier.predict(X_test), target_names=["licit", "illicit"], digits=4, zero_division=0))

    return classifier

In [25]:
print("=== Random Forest on transactions ===")
_ = random_forest_temporal(transactions_df, transactions_feature_cols)

=== Random Forest on transactions ===
TRAIN (1-29):    26,381 rows | illicit= 2,871 (10.88%)
VAL (30-34):     3,513 rows | illicit=   591 (16.82%)
TEST (35-49):    16,670 rows | illicit= 1,083 ( 6.50%)

VAL performance:
              precision    recall  f1-score   support

       licit     0.9858    0.9976    0.9917      2922
     illicit     0.9874    0.9289    0.9573       591

    accuracy                         0.9861      3513
   macro avg     0.9866    0.9633    0.9745      3513
weighted avg     0.9861    0.9861    0.9859      3513

TEST performance:
              precision    recall  f1-score   support

       licit     0.9780    0.9991    0.9884     15587
     illicit     0.9813    0.6768    0.8011      1083

    accuracy                         0.9782     16670
   macro avg     0.9796    0.8380    0.8948     16670
weighted avg     0.9782    0.9782    0.9763     16670



In [26]:
print("=== Random Forest on wallets (per-row classification, sub-sampled) ===")
_ = random_forest_temporal(wallets_df, wallets_feature_cols, sample=200_000)

=== Random Forest on wallets (per-row classification, sub-sampled) ===
TRAIN (1-29):   117,511 rows | illicit= 8,812 ( 7.50%)
VAL (30-34):    16,196 rows | illicit= 2,024 (12.50%)
TEST (35-49):    66,293 rows | illicit= 4,709 ( 7.10%)

VAL performance:
              precision    recall  f1-score   support

       licit     0.8902    0.9156    0.9027     14172
     illicit     0.2613    0.2090    0.2322      2024

    accuracy                         0.8273     16196
   macro avg     0.5757    0.5623    0.5675     16196
weighted avg     0.8116    0.8273    0.8189     16196

TEST performance:
              precision    recall  f1-score   support

       licit     0.9303    0.9515    0.9408     61584
     illicit     0.0970    0.0682    0.0801      4709

    accuracy                         0.8887     66293
   macro avg     0.5137    0.5098    0.5104     66293
weighted avg     0.8711    0.8887    0.8796     66293



## 3. Build PyG graph for transactions

Nodes = txs (all 203,769, including unknowns). Edges = `txs_edgelist` made undirected for symmetric message passing. Three masks for the chronological split. Features are standardised on the **training time steps only** to avoid leakage.

In [34]:
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, SAGEConv, GATConv
from torch_geometric.utils import to_undirected

transaction_id_to_index = {tx: i for i, tx in enumerate(transactions_df["txId"].values)}

features = transactions_df[transactions_feature_cols].astype(np.float32).values
time_steps = transactions_df["Time step"].values
labels = transactions_df["label"].values
labelled = labels != -1

scaler = StandardScaler().fit(features[labelled & (time_steps <= TRAIN_END)])
features = scaler.transform(features).astype(np.float32)

source = transactions_edges["txId1"].map(transaction_id_to_index)
target = transactions_edges["txId2"].map(transaction_id_to_index)
valid = source.notna() & target.notna()
edge_index = torch.tensor(
    np.stack([source[valid].astype(np.int64).values, target[valid].astype(np.int64).values]),
    dtype=torch.long,
)
edge_index = to_undirected(edge_index, num_nodes=len(transactions_df))

y = torch.tensor(np.where(labels < 0, 0, labels), dtype=torch.long)
train_mask = torch.tensor(labelled & (time_steps <= TRAIN_END))
val_mask   = torch.tensor(labelled & (time_steps > TRAIN_END) & (time_steps <= VAL_END))
test_mask  = torch.tensor(labelled & (time_steps > VAL_END))

graph = Data(x=torch.from_numpy(features), edge_index=edge_index, y=y)
graph.train_mask, graph.val_mask, graph.test_mask = train_mask, val_mask, test_mask
graph = graph.to(DEVICE)
print(f"nodes={graph.num_nodes:,}  edges={graph.num_edges:,}")
print(f"train={int(train_mask.sum()):,}  val={int(val_mask.sum()):,}  test={int(test_mask.sum()):,}")

nodes=203,769  edges=468,710
train=26,381  val=3,513  test=16,670


## 4. GNNs on the transaction graph

Three 2-layer architectures sharing one training loop:
- **GCN** — symmetric-normalised mean aggregation.
- **GraphSAGE** — concat self + mean of neighbours, inductive-friendly.
- **GAT** — learned attention weights per neighbour (multi-head).

Loss: weighted cross-entropy with the illicit-class weight set to `n_licit_train / n_illicit_train` to counter the ≈10:1 imbalance.

In [35]:
class GCN(nn.Module):
    def __init__(self, in_dim, hidden=64, out_dim=2, dropout=0.3):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden)
        self.conv2 = GCNConv(hidden, out_dim)
        self.dropout = dropout
    def forward(self, x, edge_index):
        h = F.relu(self.conv1(x, edge_index))
        h = F.dropout(h, p=self.dropout, training=self.training)
        return self.conv2(h, edge_index)

class GraphSAGE(nn.Module):
    def __init__(self, in_dim, hidden=64, out_dim=2, dropout=0.3):
        super().__init__()
        self.conv1 = SAGEConv(in_dim, hidden)
        self.conv2 = SAGEConv(hidden, out_dim)
        self.dropout = dropout
    def forward(self, x, edge_index):
        h = F.relu(self.conv1(x, edge_index))
        h = F.dropout(h, p=self.dropout, training=self.training)
        return self.conv2(h, edge_index)

class GAT(nn.Module):
    def __init__(self, in_dim, hidden=32, heads=4, out_dim=2, dropout=0.3):
        super().__init__()
        self.conv1 = GATConv(in_dim, hidden, heads=heads, dropout=dropout)
        self.conv2 = GATConv(hidden * heads, out_dim, heads=1, concat=False, dropout=dropout)
        self.dropout = dropout
    def forward(self, x, edge_index):
        h = F.elu(self.conv1(x, edge_index))
        h = F.dropout(h, p=self.dropout, training=self.training)
        return self.conv2(h, edge_index)

In [36]:
def train_gnn(model, graph, epochs=150, lr=1e-2, weight_decay=5e-4, log_every=50):
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    n_pos = int((graph.y[graph.train_mask] == 1).sum())
    n_neg = int((graph.y[graph.train_mask] == 0).sum())
    class_weight = torch.tensor([1.0, n_neg / max(n_pos, 1)], dtype=torch.float, device=DEVICE)

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(graph.x, graph.edge_index)
        loss = F.cross_entropy(logits[graph.train_mask], graph.y[graph.train_mask], weight=class_weight)
        loss.backward()
        optimizer.step()
        if log_every and (epoch % log_every == 0 or epoch == epochs - 1):
            print(f"  epoch {epoch:3d}  loss={loss.item():.4f}")

    model.eval()
    with torch.no_grad():
        predictions = model(graph.x, graph.edge_index).argmax(dim=-1)
    for split_name, mask in [("VAL", graph.val_mask), ("TEST", graph.test_mask)]:
        y_true = graph.y[mask].cpu().numpy()
        y_pred = predictions[mask].cpu().numpy()
        print(f"{split_name}:")
        print(classification_report(y_true, y_pred, target_names=["licit", "illicit"], digits=4, zero_division=0))
    return model

in_dim = graph.num_node_features
for name, constructor in [("GCN", GCN), ("GraphSAGE", GraphSAGE), ("GAT", GAT)]:
    print(f"\n=== {name} ===")
    train_gnn(constructor(in_dim), graph, epochs = 300)


=== GCN ===
  epoch   0  loss=0.9125
  epoch  50  loss=0.1890
  epoch 100  loss=0.1430
  epoch 150  loss=0.1171
  epoch 200  loss=0.1049
  epoch 250  loss=0.0920
  epoch 299  loss=0.0863
VAL:
              precision    recall  f1-score   support

       licit     0.9780    0.9285    0.9526      2922
     illicit     0.7172    0.8968    0.7970       591

    accuracy                         0.9231      3513
   macro avg     0.8476    0.9126    0.8748      3513
weighted avg     0.9341    0.9231    0.9264      3513

TEST:
              precision    recall  f1-score   support

       licit     0.9736    0.9533    0.9633     15587
     illicit     0.4830    0.6279    0.5460      1083

    accuracy                         0.9322     16670
   macro avg     0.7283    0.7906    0.7547     16670
weighted avg     0.9417    0.9322    0.9362     16670


=== GraphSAGE ===
  epoch   0  loss=0.7552
  epoch  50  loss=0.0841
  epoch 100  loss=0.0465
  epoch 150  loss=0.0364
  epoch 200  loss=0.0290
  e

## 5. Temporal Transformer on wallet feature sequences

Each labelled wallet appears in 1–49 time steps. We treat the wallet's sequence of per-time-step feature vectors as a token stream and apply a small Transformer encoder with a learned positional embedding keyed by the **absolute** `Time step` index (not by ordinal sequence position — the index has block-time meaning), then masked mean-pool and classify.

Why this is interesting:
- Self-attention can relate distant time steps directly, unlike a GRU.
- The positional embedding captures real calendar drift, not just sequence order.

A wallet is assigned to a split by its **latest** observed time step: train if max ≤ 29, val if 30–34, test if ≥ 35. Wallets with mixed activity end up in the latest split; this is an approximation acceptable for an exploration baseline.

In [30]:
from torch.utils.data import DataLoader, TensorDataset

labelled_wallets = wallets_df[wallets_df["label"] != -1].copy()
scaler = StandardScaler().fit(
    labelled_wallets.loc[labelled_wallets["Time step"] <= TRAIN_END, wallets_feature_cols].values
)
labelled_wallets[wallets_feature_cols] = (
    scaler.transform(labelled_wallets[wallets_feature_cols].values).astype(np.float32)
)

rng = np.random.default_rng(42)
address_label = labelled_wallets.groupby("address")["label"].first()
illicit_addresses = address_label[address_label == 1].index.values
licit_addresses = address_label[address_label == 0].index.values
n_licit_keep = min(40_000, len(licit_addresses))
chosen_addresses = np.concatenate([
    illicit_addresses,
    rng.choice(licit_addresses, size=n_licit_keep, replace=False),
])
subset = (labelled_wallets[labelled_wallets["address"].isin(set(chosen_addresses))]
          .sort_values(["address", "Time step"]))

T_MAX = 12
feature_dim = len(wallets_feature_cols)

def assign_split(max_time_step):
    if max_time_step <= TRAIN_END:
        return 0
    if max_time_step <= VAL_END:
        return 1
    return 2

sequence_features, sequence_times, sequence_masks, sequence_labels, sequence_split = [], [], [], [], []
for _, group in subset.groupby("address", sort=False):
    group = group.iloc[-T_MAX:]
    n = len(group)
    x = np.zeros((T_MAX, feature_dim), np.float32)
    t = np.zeros(T_MAX, np.int64)
    pad_mask = np.ones(T_MAX, dtype=bool)
    x[:n] = group[wallets_feature_cols].values
    t[:n] = group["Time step"].values
    pad_mask[:n] = False
    sequence_features.append(x); sequence_times.append(t); sequence_masks.append(pad_mask)
    sequence_labels.append(int(group["label"].iloc[0]))
    sequence_split.append(assign_split(int(group["Time step"].max())))

X_sequences = torch.tensor(np.stack(sequence_features))
T_sequences = torch.tensor(np.stack(sequence_times))
M_sequences = torch.tensor(np.stack(sequence_masks))
Y_sequences = torch.tensor(sequence_labels, dtype=torch.long)
split_tensor = torch.tensor(sequence_split, dtype=torch.long)

train_idx = (split_tensor == 0)
val_idx   = (split_tensor == 1)
test_idx  = (split_tensor == 2)
print(f"sequences={X_sequences.shape}  "
      f"train={int(train_idx.sum())}  val={int(val_idx.sum())}  test={int(test_idx.sum())}")

sequences=torch.Size([54266, 12, 55])  train=29959  val=5026  test=19281


In [31]:
class TemporalTransformer(nn.Module):
    def __init__(self, in_dim, d_model=64, n_heads=4, n_layers=2, dropout=0.2):
        super().__init__()
        self.projection = nn.Linear(in_dim, d_model)
        self.position = nn.Embedding(50, d_model)  # ts in 1..49; index 0 reserved for padding
        encoder_layer = nn.TransformerEncoderLayer(
            d_model, n_heads, dim_feedforward=128, dropout=dropout, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, n_layers)
        self.classifier = nn.Linear(d_model, 2)
    def forward(self, x, time_step, padding_mask):
        h = self.projection(x) + self.position(time_step)
        h = self.encoder(h, src_key_padding_mask=padding_mask)
        valid = (~padding_mask).unsqueeze(-1).float()
        pooled = (h * valid).sum(dim=1) / valid.sum(dim=1).clamp(min=1)
        return self.classifier(pooled)

In [33]:
def make_loader(idx, batch_size, shuffle):
    return DataLoader(
        TensorDataset(X_sequences[idx], T_sequences[idx], M_sequences[idx], Y_sequences[idx]),
        batch_size=batch_size, shuffle=shuffle,
    )

train_loader = make_loader(train_idx, batch_size=512, shuffle=True)
val_loader   = make_loader(val_idx,   batch_size=1024, shuffle=False)
test_loader  = make_loader(test_idx,  batch_size=1024, shuffle=False)

model = TemporalTransformer(feature_dim).to(DEVICE)
n_pos = int((Y_sequences[train_idx] == 1).sum())
n_neg = int((Y_sequences[train_idx] == 0).sum())
class_weight = torch.tensor([1.0, n_neg / max(n_pos, 1)], dtype=torch.float, device=DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(100):
    model.train()
    running_loss, seen = 0.0, 0
    for x_batch, t_batch, mask_batch, y_batch in train_loader:
        x_batch, t_batch = x_batch.to(DEVICE), t_batch.to(DEVICE)
        mask_batch, y_batch = mask_batch.to(DEVICE), y_batch.to(DEVICE)
        optimizer.zero_grad()
        loss = F.cross_entropy(model(x_batch, t_batch, mask_batch), y_batch, weight=class_weight)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * y_batch.size(0)
        seen += y_batch.size(0)
    if epoch % 5 == 0 or epoch == 19:
        print(f"  epoch {epoch:2d}  loss={running_loss / seen:.4f}")

def evaluate(loader, name):
    model.eval()
    y_true_list, y_pred_list = [], []
    with torch.no_grad():
        for x_batch, t_batch, mask_batch, y_batch in loader:
            x_batch, t_batch = x_batch.to(DEVICE), t_batch.to(DEVICE)
            mask_batch = mask_batch.to(DEVICE)
            y_pred_list.append(model(x_batch, t_batch, mask_batch).argmax(dim=-1).cpu().numpy())
            y_true_list.append(y_batch.numpy())
    y_true = np.concatenate(y_true_list)
    y_pred = np.concatenate(y_pred_list)
    print(f"{name}:")
    print(classification_report(y_true, y_pred, target_names=["licit", "illicit"],
                                digits=4, zero_division=0))

evaluate(val_loader,  "VAL  (in-distribution)")
evaluate(test_loader, "TEST (the cliff)")

  epoch  0  loss=0.4377
  epoch  5  loss=0.2679
  epoch 10  loss=0.2524
  epoch 15  loss=0.2387
  epoch 19  loss=0.2265
  epoch 20  loss=0.2232
  epoch 25  loss=0.2101
  epoch 30  loss=0.1996
  epoch 35  loss=0.1910
  epoch 40  loss=0.1843
  epoch 45  loss=0.1765
  epoch 50  loss=0.1727
  epoch 55  loss=0.1693
  epoch 60  loss=0.1695
  epoch 65  loss=0.1664
  epoch 70  loss=0.1635
  epoch 75  loss=0.1607
  epoch 80  loss=0.1578
  epoch 85  loss=0.1570
  epoch 90  loss=0.1569
  epoch 95  loss=0.1540
VAL  (in-distribution):
              precision    recall  f1-score   support

       licit     0.6783    0.8019    0.7349      3240
     illicit     0.4632    0.3102    0.3716      1786

    accuracy                         0.6271      5026
   macro avg     0.5708    0.5560    0.5532      5026
weighted avg     0.6019    0.6271    0.6058      5026

TEST (the cliff):
              precision    recall  f1-score   support

       licit     0.7880    0.7503    0.7687     14343
     illicit     0

## Where to go next
- **Heterogeneous GNN on the wallet sub-dataset** — a `HeteroData` graph with wallet & tx node types and the three edge types (`AddrAddr`, `AddrTx`, `TxAddr`); RGCN / HAN / HGT.
- **EvolveGCN / TGN** — let GCN parameters drift over time so message passing reflects the active sub-graph at each time step.
- **Self-supervised pre-training** on unknown nodes (≈77% of tx, ≈68% of wallets) before fine-tuning on labels.
- **Calibrated thresholds** — tune the decision threshold on val and apply to test; F1 of the illicit class is very threshold-sensitive at this imbalance ratio.
- **Cliff diagnostics** — break test metrics down by time step to localise where each model collapses (Weber et al. observed an abrupt drop near step 43).